In [1]:
import os
import pandas as pd

def parse_log_file(log_file, eval_key='CNN'):
    is_valid = True
    result_dict = {
        "model_name": "",
        "dataset": "",
        "prefix": "",
        "seed": "",
        "epochs": 0,
        "dot_epochs": 0,
        "bcb_lrscale": 0.01,
        "average_accuracy_all": 0.0,
        "last_accuracy_all": 0.0,
        "last_accuracy_in": 0.0,
        "last_accuracy_out": 0.0,
        "last_accuracy_out_worst": 0.0,
        "forgetting": -1e-5,
    }
    
    with open(log_file, 'r') as f:
        lines = f.readlines()
    
    task_reference_domains = {}
    class_id_pairs = []
    domain_accuracies = {}
    no_nme = False
    
    for line in lines:
        if "No NME" in line and eval_key == "NME":
            no_nme = True
        if "[trainer.py] => model_name:" in line:
            result_dict["model_name"] = line.split(":")[-1].strip()
        if "[trainer.py] => prefix" in line:
            result_dict["prefix"] = line.split(":")[-1].strip()
        if "[trainer.py] => seed:" in line:
            result_dict["seed"] = line.split(":")[-1].strip()
        if "[trainer.py] => dataset:" in line:
            result_dict["dataset"] = line.split(":")[-1].strip()
        if "[trainer.py] => epochs:" in line:
            result_dict["epochs"] = int(line.split(":")[-1].strip())
        if "[trainer.py] => dot_epochs:" in line:
            result_dict["dot_epochs"] = int(line.split(":")[-1].strip())
        if "[trainer.py] => bcb_lrscale:" in line:
            result_dict["bcb_lrscale"] = float(line.split(":")[-1].strip())
        if f"[trainer.py] => Average Accuracy ({eval_key}):" in line:
            result_dict["average_accuracy_all"] = float(line.split(":")[-1].strip())
        if f"[trainer.py] => Last Accuracy ({eval_key}):" in line:
            result_dict["last_accuracy_all"] = float(line.split(":")[-1].strip())
        if (f"[trainer.py] => {eval_key}"+": {'total':" )in line:
            cnn_dict = eval(line.split(f"{eval_key}:")[-1].strip())
            result_dict["last_accuracy_all"] = cnn_dict.get("total", 0.0)
        if f"[trainer.py] => Forgetting ({eval_key}):" in line:
            result_dict["forgetting"] = float(line.split(":")[-1].strip())
        if "=> Task" in line and "reference domain is" in line:
            task_id = int(line.split("Task")[1].split(":")[0].strip())
            domain_part = line.split("reference domain is")[-1].strip()
            domain_id = int(domain_part.split("[")[1].split("]")[0].strip())
            task_reference_domains[task_id] = domain_id
        if "[trainer.py] => Class ID pairs:" in line:
            class_id_pairs = eval(line.split(":")[-1].strip())
        if "[trainer.py] => Domain [" in line and f"{eval_key}:" in line:
            domain_part = line.split("[trainer.py] => Domain")[-1].strip()
            domain_id = int(domain_part.split("[")[1].split("]")[0].strip())
            accuracies = eval(line.split(f"{eval_key}:")[-1].strip())
            domain_accuracies[domain_id] = accuracies

    # assert result_dict["forgetting"] > 0.0, f"invalid log file: {log_file}"
    if no_nme:
        print(f"No NME for {log_file}")
        is_valid = False
    elif result_dict["forgetting"] < 0.0:
        print(f"incomplete log file: {log_file}")
        is_valid = False
    
    if not is_valid:
        return result_dict, False

    # Calculate last_accuracy_in, last_accuracy_out, and last_accuracy_out_worst
    in_accuracies = []
    out_accuracies = []
    out_worst_accuracies = []
    
    for task_id, ref_domain in task_reference_domains.items():
        class_pair = class_id_pairs[task_id]
        class_key = f"{class_pair[0]:02d}-{class_pair[1]:02d}"
        
        # In-domain accuracy
        in_acc = domain_accuracies[ref_domain].get(class_key, 0.0)
        in_accuracies.append(in_acc)
        
        # Out-domain accuracy
        out_domains = [domain for domain in domain_accuracies.keys() if domain != ref_domain]
        out_accs = [domain_accuracies[domain].get(class_key, 0.0) for domain in out_domains]
        out_accuracies.append(sum(out_accs) / len(out_accs))
        
        # Worst out-domain accuracy
        out_worst_accuracies.append(min(out_accs))
    
    result_dict["last_accuracy_in"] = sum(in_accuracies) / len(in_accuracies) if in_accuracies else 0.0
    result_dict["last_accuracy_out"] = sum(out_accuracies) / len(out_accuracies) if out_accuracies else 0.0
    result_dict["last_accuracy_out_worst"] = sum(out_worst_accuracies) / len(out_worst_accuracies) if out_worst_accuracies else 0.0
    
    return result_dict, is_valid


def parse_log_folder(log_folder, eval_key='CNN'):

    all_result = []

    for root, dirs, files in os.walk(log_folder):
        for file in files:
            if file.endswith(".log"):
                log_file = os.path.join(root, file)
                result_dict, is_valid = parse_log_file(log_file, eval_key)
                if is_valid: all_result.append(result_dict)

    # Rearrange the contents of the csv file in the following form:
    # method | prefix | seed | dataset | average_accuracy_all | last_accuracy_all | last_accuracy_in | last_accuracy_out | last_accuracy_out_worst | forgetting
    df = pd.DataFrame(all_result)
    df = df[["model_name", "prefix", "seed", "epochs", "dot_epochs", "bcb_lrscale", "dataset", "average_accuracy_all", "last_accuracy_all", "last_accuracy_in", "last_accuracy_out", "last_accuracy_out_worst", "forgetting"]]
    df = df.sort_values(by=["model_name", "dataset", "prefix", "seed"])
    df.to_csv(f"{eval_key}_results.csv", index=False)


    # Filter seeds to include only [1994, 1995, 1996]
    valid_seeds = ["1994", "1995", "1996"]
    df_filtered = df[df["seed"].astype(str).isin(valid_seeds)]

    # valid_model_name = ["dot"]
    # df_filtered = df_filtered[df_filtered["model_name"].isin(valid_model_name)]

    # Save the filtered results to a new CSV file
    df_filtered.to_csv(f"{eval_key}_results_filtered.csv", index=False)

    # Calculate mean and std for each setup (grouped by model_name, prefix, dataset) for the filtered seeds
    mean_std_df = df_filtered.groupby(["model_name", "prefix", "epochs", "dot_epochs", "bcb_lrscale", "dataset"]).agg({
        "average_accuracy_all": ["mean", "std"],
        "last_accuracy_all": ["mean", "std"],
        "last_accuracy_in": ["mean", "std"],
        "last_accuracy_out": ["mean", "std"],
        "last_accuracy_out_worst": ["mean", "std"],
        "forgetting": ["mean", "std"],
    }).reset_index()


    # Flatten the multi-level columns
    mean_std_df.columns = ["_".join(col).strip() for col in mean_std_df.columns.values]

    # Save the mean and std results to a new CSV file
    mean_std_df.to_csv(f"{eval_key}_results_mean_std_filtered.csv", index=False)


root_folder = "/gpfs-flash/junlab/liyuan/hongwei/DGIL"
parse_log_folder(root_folder, eval_key='CNN')
# parse_log_folder(root_folder, eval_key='NME')